# Colab Worker, Model-Selection Loop

Run this notebook in a Colab GPU runtime for real regression model-selection cycles. It uses `experiments/colab_worker_model_loop.json`, which runs model-only candidates from existing cached feature CSVs and pushes only compact summary artifacts back to GitHub.

## 1. Clone or Update Repository

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
REPO_DIR = Path("/content/Aberration-Simulation")

def load_github_token():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            return token.strip(), "Colab Secrets"
    except Exception:
        pass

    token = os.environ.get("GITHUB_TOKEN")
    if token:
        return token.strip(), "runtime environment"

    print("Paste your fine-grained GitHub token into the input box for this cell.")
    print("It is used only for this Colab runtime and is not written to the notebook file.")
    token = input("GITHUB_TOKEN: ").strip()
    if token:
        return token, "manual notebook input"
    return "", "missing"

token, token_source = load_github_token()
if not token:
    raise RuntimeError("GITHUB_TOKEN was not provided; cannot push worker artifacts to GitHub.")
os.environ["GITHUB_TOKEN"] = token
print(f"GITHUB_TOKEN loaded from {token_source}.")

if REPO_DIR.exists():
    subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Check GPU and Cached Data

In [ ]:
!nvidia-smi
!python scripts/check_backend.py
from pathlib import Path
cached_feature_csvs = sorted(Path("training_results").glob("**/training_features_enhanced.csv"))
cached_feature_csvs += sorted(Path("training_results").glob("**/training_features_raw_angles.csv"))
print("cached feature CSVs:")
for path in cached_feature_csvs[-20:]:
    print(" ", path)
if not cached_feature_csvs:
    raise RuntimeError("No cached feature CSV found under training_results/. This model loop is configured to use existing CSVs only and will not regenerate simulations.")


## 3. Start Model-Selection Worker Loop

Run this cell once. After each cycle, the worker pushes compact metrics/manifests and waits for Codex to push the next changed config fingerprint before continuing.

In [ ]:
WORKER_CONFIG = "experiments/colab_worker_model_loop.json"
WORKER_CYCLES = 8

!python scripts/colab_regression_worker.py --config {WORKER_CONFIG} --cycles {WORKER_CYCLES}
